In [1]:
# %load_ext cudf.pandas

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


# Analysis of NYC Flight Trends in Python


In [3]:
import numpy as np
import pandas as pd
import time

In [4]:
start_time = time.time()

In [5]:
### cell 0 ###

flights_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_flights.csv"
)
factor = 20
flights_df = pd.concat([flights_df] * factor)

In [6]:
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

((6735520, 19),
 Index(['year', 'month', 'day', 'dep_time', 'sched_dep_time', 'dep_delay',
        'arr_time', 'sched_arr_time', 'arr_delay', 'carrier', 'flight',
        'tailnum', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute',
        'time_hour'],
       dtype='object'),
 year                int64
 month               int64
 day                 int64
 dep_time          float64
 sched_dep_time      int64
 dep_delay         float64
 arr_time          float64
 sched_arr_time      int64
 arr_delay         float64
 carrier            object
 flight              int64
 tailnum            object
 origin             object
 dest               object
 air_time          float64
 distance            int64
 hour                int64
 minute              int64
 time_hour          object
 dtype: object)

In [7]:
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z
5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,1696,N39463,EWR,ORD,150.0,719,5,58,2013-01-01T10:00:00Z
6,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,507,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01T11:00:00Z
7,2013,1,1,557.0,600,-3.0,709.0,723,-14.0,EV,5708,N829AS,LGA,IAD,53.0,229,6,0,2013-01-01T11:00:00Z
8,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,79,N593JB,JFK,MCO,140.0,944,6,0,2013-01-01T11:00:00Z
9,2013,1,1,558.0,600,-2.0,753.0,745,8.0,AA,301,N3ALAA,LGA,ORD,138.0,733,6,0,2013-01-01T11:00:00Z


## How many flights were there from NYC airports to Seattle in 2013?

In [8]:
### cell 3 ###

flights_df["dest"][flights_df["dest"] == "SEA"].value_counts()

dest
SEA    78460
Name: count, dtype: int64

--There were a total of 3923 flights from NYC to Seattle in 2013--

## How many airlines fly from NYC to Seattle?

In [9]:
### cell 4 ###

flights_df["carrier"][flights_df["dest"] == "SEA"].value_counts()

carrier
DL    24260
UA    22340
AS    14280
B6    10280
AA     7300
Name: count, dtype: int64

There are a total of 5 airline carriers that fly from NYC to Seattle. The total number of flights for each of the airline carriers is as follows:
DL - 1213, UA- 1117, AS-714, B6-514, AA-365

## How many unique air planes fly from NYC to Seattle?

In [10]:
### cell 5 ###

len(flights_df["tailnum"][flights_df["dest"] == "SEA"].unique())

936

There are a total of 936 unique air planes from NYC to Seattle.

## What is the average arrival delay for flights from NC to Seattle?

In [11]:
### cell 6 ###

flights_df["arr_delay"][flights_df["dest"] == "SEA"].mean()

-1.0990990990990992

The average arrival delay for flights from NYC to Seattle is -1.099. The negative value implies that instead of a delay, flights arrive before the scheduled time by 1.099 units.

## What proportion of flights to Seattle come from each NYC airport?

In [12]:
### cell 7 ###

f = flights_df[flights_df["dest"] == "SEA"].groupby("origin").size()
f_total = len(flights_df[flights_df.dest == "SEA"])
f.loc["EWR"] / f_total, f.loc["JFK"] / f_total

(0.46673464185572267, 0.5332653581442773)

Total flights from EWR to Seattle: 1810
Proportion of flights from EWR to Seattle: 0.46
Total flights from JFK to Seattle: 2075
Proportion of flights from JFK to Seattle: 0.53

## Flights are often delayed. The following questions explore delay patterns:
## Which date has the largest average departure delay? Which date has the largest average arrival delay?

In [13]:
### cell 8 ###

df = flights_df.groupby(["month", "day"], as_index=False).agg({"dep_delay": np.mean})
df2 = flights_df.groupby(["month", "day"], as_index=False).agg({"arr_delay": np.mean})
df.loc[df["dep_delay"].idxmax()], df2.loc[df2["arr_delay"].idxmax()]

/var/tmp/ipykernel_30147/3494824320.py:3: FutureWarning: The provided callable <function mean at 0x7f418811ca60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df = flights_df.groupby(['month', 'day'], as_index=False).agg({'dep_delay': np.mean})
/var/tmp/ipykernel_30147/3494824320.py:4: FutureWarning: The provided callable <function mean at 0x7f418811ca60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df2 = flights_df.groupby(['month', 'day'], as_index=False).agg({'arr_delay': np.mean})


(month         3.000000
 day           8.000000
 dep_delay    83.536921
 Name: 66, dtype: float64,
 month         3.000000
 day           8.000000
 arr_delay    85.862155
 Name: 66, dtype: float64)

The date 8/3/2013 (8th March 2013) has the largest average departure delay equal to 83.53 units.
The date 8/3/2013 (8th March 2013) has the largest arrival delay, equal to 85.86 units.

## What was the worst day to fly out of NYC in 2013 if you dislike delayed flights?


In [14]:
### cell 9 ###

df

,month,day,dep_delay
0,1,1,11.548926
1,1,2,13.858824
2,1,3,10.987832
3,1,4,8.951595
4,1,5,5.732218
...,...,...,...
360,12,27,10.937630
361,12,28,7.981550
362,12,29,22.309551
363,12,30,10.698113


In [15]:
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

/var/tmp/ipykernel_30147/2706612504.py:3: FutureWarning: The provided callable <function mean at 0x7f418811ca60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df = flights_df.groupby(['day', 'month'], as_index=False).agg({'arr_delay': np.mean, 'dep_delay': np.mean})


,day,month,arr_delay,dep_delay,total_delay
86,8,3,85.862155,83.536921,169.399077


The worst day to fly out of NYC in 2013 was 8th March 2013. The average arrival delay was 85.86units, the average departure delay was 83.54 units and the total delay was 169 units.

## Are there any seasonal patterns in departure delays for flights from NYC?

In [16]:
### cell 11 ###

ds = flights_df.dropna(subset=["dep_delay"]).groupby(["month"])["dep_delay"].mean()
ds

month
1     10.036665
2     10.816843
3     13.227076
4     13.938038
5     12.986859
6     20.846332
7     21.727787
8     12.611040
9      6.722476
10     6.243988
11     5.435362
12    16.576688
Name: dep_delay, dtype: float64

The average departure delay gradually increases from the month of January to July, followed by a decrease from August to November. Finally, there is an increase from December again. This signifies that certain seasonal factors may be responsible for decreasing departure delays in Fall and increasing departure delays in late Winter, Spring and Summer. Since the peak departure delays occur in Summer(June, July), other factors such as increased passenger traffic at airports (for holidaying) need to be explored for further analysis. 

## On average, how do departure delays vary over the course of a day?

In [17]:
### cell 12 ###

dt = flights_df.dropna(subset=["dep_delay"]).groupby(["hour"])["dep_delay"].mean()
dt

hour
5      0.687757
6      1.642796
7      1.914078
8      4.127948
9      4.583738
10     6.498295
11     7.191650
12     8.614849
13    11.437650
14    13.818874
15    16.894565
16    18.757017
17    21.100606
18    21.110082
19    24.784791
20    24.304105
21    24.195743
22    18.791097
23    14.017176
Name: dep_delay, dtype: float64

There is a rapid increase in departure delays betwen 12 AM and 3 AM followed by a steep decline from 3 AM to 4 AM. Again, from 4 AM onwards, there is a gradual rise till 9 PM, forming a small peak around 11 PM. Finally, the departure delay starts decreasing from 11 PM to 12 AM. This may be attributed to the assumption that there are fewer flights at the timeslot of 11 PM to 12 AM which causes the rapid decline in departure delays. 

## Which flight departing NYC in 2013 flew the fastest?

In [18]:
### cell 13 ###

df = flights_df
df["speed"] = df["distance"] / df["air_time"]
df[df["speed"] == df.speed.max()]

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,speed
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077
216447,2013,5,25,1709.0,1700,9.0,1923.0,1937,-14.0,DL,1499,N666DN,LGA,ATL,65.0,762,17,0,2013-05-25T21:00:00Z,11.723077


The flight 1499 with tailnum N666DN belonging to carrier DL travelling from LGA to ATL has the highest speed. 
It covers a distance of 762 miles in 65 minutes and has a speed of 11.723 mph.

## Which flights (i.e. carrier + flight + dest) happen every day? Where do they fly to?

In [19]:
### cell 14 ###

count = len(flights_df)
df = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="Size")
for i in df.index:
    if df.loc[i]["Size"] == 365:
        print(
            "Carrier: %s, Flight: %s, Destination: %s"
            % (df.loc[i]["carrier"], df.loc[i]["flight"], df.loc[i]["dest"])
        )

In [20]:
### cell 15 ###

gdf = flights_df
counts = gdf.groupby(["carrier", "flight", "dest"]).size().reset_index(name="size")
complete_year = counts[counts["size"] == 365]
complete_year[["carrier", "flight", "dest"]]

,carrier,flight,dest


All the above 18 flights fly out of NYC every day.

## Which airline carrier has the best service in terms of the lowest average flight arrival and departure delays in June 2013?

In [21]:
### cell 16 ###

df = flights_df[flights_df["month"] == 6]
df = df.groupby("carrier", as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
for i in df.index:
    if df.loc[i]["total_delay"] == df["total_delay"].min():
        print(df.loc[i]["carrier"], df.loc[i]["total_delay"])
df

HA 3.3


/var/tmp/ipykernel_30147/173071645.py:4: FutureWarning: The provided callable <function mean at 0x7f418811ca60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df = df.groupby('carrier', as_index=False).agg({'arr_delay': np.mean, 'dep_delay': np.mean})
/var/tmp/ipykernel_30147/173071645.py:4: FutureWarning: The provided callable <function mean at 0x7f418811ca60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  df = df.groupby('carrier', as_index=False).agg({'arr_delay': np.mean, 'dep_delay': np.mean})


,carrier,arr_delay,dep_delay,total_delay
0,9E,22.511905,28.952978,51.464883
1,AA,6.481178,14.627778,21.108956
2,AS,-3.750000,13.083333,9.333333
3,B6,18.541319,20.392170,38.933488
4,DL,13.261829,18.735941,31.997770
5,EV,21.267975,25.496834,46.764809
6,F9,26.636364,29.436364,56.072727
7,FL,41.966805,38.806584,80.773389
8,HA,1.833333,1.466667,3.300000
9,MQ,23.143722,20.842342,43.986064


The carrier Hawaiian Airlines(HA) has the best service in terms of lowest average arrival and departure delays in June 2013. The lowest total delay is 3.3 units. The carrier with the highest average total delay (arrival delay + departure delay) is OO with an average total delay of 129.5 units. Thus, we can classify SkyWest Airlines (OO) as the worst carrier in terms of average total delay.

From the plot, we can see that the total delay (arrival delay + departure delay) for Hawaiian Airlines (HA) is the lowest. The median and quartile range for United Airlines is the lowest among all the other carriers.  The median and quartile range for Skywest Airlines(OO) is the highest from each of the plots. 

## What weather conditions are associated with flight delays leaving NYC? Use graphics to explore.

In [22]:
### cell 17 ###

weather_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)

,year_x,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,temp,dewp,humid,wind_dir,wind_speed,wind_gust,precip,pressure,visib,time_hour_y
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,...,39.92,24.98,54.81,250.0,14.96014,21.86482,0.0,1011.4,10.0,2013-01-01T10:00:00Z
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,...,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0,2013-01-01T10:00:00Z
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,...,39.02,26.96,61.63,260.0,14.96014,NaN,0.0,1012.1,10.0,2013-01-01T10:00:00Z
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,...,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0,2013-01-01T11:00:00Z
5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,...,39.02,28.04,64.43,260.0,12.65858,NaN,0.0,1011.9,10.0,2013-01-01T10:00:00Z
6,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,...,37.94,28.04,67.21,240.0,11.50780,NaN,0.0,1012.4,10.0,2013-01-01T11:00:00Z
7,2013,1,1,557.0,600,-3.0,709.0,723,-14.0,EV,...,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0,2013-01-01T11:00:00Z
8,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,...,37.94,26.96,64.29,260.0,13.80936,NaN,0.0,1012.6,10.0,2013-01-01T11:00:00Z
9,2013,1,1,558.0,600,-2.0,753.0,745,8.0,AA,...,39.92,24.98,54.81,260.0,16.11092,23.01560,0.0,1011.7,10.0,2013-01-01T11:00:00Z


In [23]:
end_time = time.time()
print(end_time - start_time)

14.173023223876953


After merging the flights and weather dataframes based on the columns month, day, hour and origin, two separate graphs have been plotted:

1. Departure Delay Vs. Humidity:
The Departure delay and Humidity plot indicates that there is a rise in departure delay as humidity increases till Humidity reaches the 50th unit. The departure delay reduces when humidity is between roughly 55-65 units. The graph is more or less constant except for the dip at the end. There are a lot of outliers. This signifies that there can be significantly high departure delays even when the humidity is not that high or low. This signifies that other factors such as wind speed, temperature, precipitation may be responsible for the numerous outliers. 

2. Arrival Delay Vs. Temperature:
The Arrival Delay and Temperature plot also indicates that there is a rise in arrival delay as temperature increases till roughly 35 units. There is a huge spike in Arrival delay at the 24-25th unit followed by the biggest spike in the 40th unit. The graph then remains more or less constant execpt for the dip at the end and a few spikes at the 45-55th unit, 64th, 85th and 9th units. Due to the significant dip in arrival delay at very high temperatures (90 units and above), we cannot conclude that there is a correlation between arrival delay and temperature. However, other factors may be responsible for the fluctuating arrival delay and the various spikes seen in the graph that we probably have not accounted for. Other factors not related to weather, such as waiting time at airports, passenger traffic, immigration queues etc. might have a significant effect on arrival delay which is beyond the purview of our dataset.